# Step 1 — weighted XGBoost versus SMOTE

Compare the current weighted baseline with four SMOTE ratios. Keep all other settings fixed and evaluate at threshold 0.5 first. This notebook runs five fits, sequentially; it never loads test arrays. Run from the project root (or its root-level notebook).

SMOTE runs only during pipeline fitting, after training-only feature selection. Ordinary SMOTE can produce fractional binary/one-hot values; treat this as an experiment on the existing engineered representation. The saved pipeline expects 44 engineered features, not raw customer records.

Current validation is used for selection. Historical code tuned on the older holdout, so the current test subset must not be described as historically untouched. Do not regenerate splits here.

If needed, install dependencies in this notebook's kernel with `%pip install -r requirements.txt`, then restart the kernel. Full-data SMOTE can consume significant memory; candidates run one at a time.


In [ ]:
from pathlib import Path
import hashlib
import platform
from importlib.metadata import version
from time import perf_counter

import numpy as np

import pandas as pd
import mlflow

from mlflow.models import infer_signature
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from xgboost import XGBClassifier

ROOT = Path.cwd()
DATA_PATH = ROOT / "data/processed_datasets.npz"
assert DATA_PATH.exists(), "Open this notebook from the project root."
with np.load(DATA_PATH, allow_pickle=False) as data:
    X_train = data["X_train"]
    X_val = data["X_val"]
    Y_train = data["Y_train"]
    Y_val = data["Y_val"]

assert X_train.shape[1] == X_val.shape[1] >= 25
for X, y in ((X_train, Y_train), (X_val, Y_val)):
    assert len(X) == len(y)
    assert np.isfinite(X).all()
    assert set(np.unique(y)) == {0, 1}
print("Training:", X_train.shape, "Validation:", X_val.shape)
print("Training class counts:", dict(zip(*np.unique(Y_train, return_counts=True))))


Training: (659028, 44) Validation: (82379, 44)
Training class counts: {np.int8(0): np.int64(593685), np.int8(1): np.int64(65343)}


In [4]:
SEED = 42
N_FEATURES = 25
THRESHOLD = 0.5
MODEL_PARAMS = dict(
    learning_rate=0.03, max_depth=3, n_estimators=400,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, eval_metric="logloss", n_jobs=-1,
)
CANDIDATES = [("weighted_baseline", None, 7)] + [
    (f"smote_{ratio:.2f}", ratio, 1) for ratio in (0.25, 0.50, 0.75, 1.00)
]

def make_pipeline(ratio, weight):
    sampler = "passthrough" if ratio is None else SMOTE(
        sampling_strategy=ratio, random_state=SEED, k_neighbors=5,
    )
    return Pipeline([
        ("feature_selection", SelectKBest(f_classif, k=N_FEATURES)),
        ("smote", sampler),
        ("model", XGBClassifier(**MODEL_PARAMS, scale_pos_weight=weight)),
    ])

def evaluate(y, scores):
    predictions = (scores >= THRESHOLD).astype(int)
    return {
        "average_precision": average_precision_score(y, scores),
        "f1": f1_score(y, predictions, zero_division=0),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "accuracy": accuracy_score(y, predictions),
    }

# Hash the existing file without changing it or materializing its test arrays.
hasher = hashlib.sha256()
with DATA_PATH.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        hasher.update(chunk)
data_hash = hasher.hexdigest()
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow_churn.db'}")
mlflow.set_experiment("churn-xgboost-smote-comparison")
print("Ready:", len(CANDIDATES), "fits; threshold:", THRESHOLD)


2026/09/29 14:36:21 INFO mlflow.tracking.fluent: Experiment with name 'churn-xgboost-smote-comparison' does not exist. Creating a new experiment.


Ready: 5 fits; threshold: 0.5


## Run the comparison

The next cell performs the full training. Each candidate gets one MLflow run, its fitted pipeline, validation probabilities for later threshold comparison, and metrics at 0.5. Existing experiments are preserved. Re-running creates new runs.


In [5]:
import tempfile

results = []
validation_probabilities = {}
for name, ratio, weight in CANDIDATES:
    print("Training:", name, flush=True)
    pipeline = make_pipeline(ratio, weight)
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({
            **MODEL_PARAMS,
            "model_type": "xgboost", "n_features": N_FEATURES,
            "scale_pos_weight": weight,
            "sampling_strategy": "none" if ratio is None else ratio,
            "smote_k_neighbors": "none" if ratio is None else 5,
            "decision_threshold": THRESHOLD, "evaluation_split": "validation",
            "dataset_sha256": data_hash,
            "train_rows": len(Y_train), "validation_rows": len(Y_val),
        })
        mlflow.set_tags({
            "input_representation": "existing_engineered_features",
            "test_history": "older_holdout_used_for_selection",
        })
        mlflow.log_dict({
            "python": platform.python_version(),
            **{pkg: version(pkg) for pkg in (
                "numpy", "pandas", "scikit-learn", "imbalanced-learn", "xgboost", "mlflow"
            )},
        }, "environment.json")
        started = perf_counter()
        pipeline.fit(X_train, Y_train)
        fit_seconds = perf_counter() - started
        scores = pipeline.predict_proba(X_val)[:, 1]
        metrics = evaluate(Y_val, scores)
        mlflow.log_metrics({**metrics, "fit_seconds": fit_seconds})
        mlflow.sklearn.log_model(
            pipeline, name="model", serialization_format="cloudpickle",
            input_example=X_train[:5],
            signature=infer_signature(X_train[:5], pipeline.predict(X_train[:5])),
        )
        mlflow.log_dict({"indices": pipeline.named_steps[
            "feature_selection"
        ].get_support(indices=True).tolist()}, "selected_features.json")
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "validation_predictions.npz"
            np.savez_compressed(path, y=Y_val, probabilities=scores)
            mlflow.log_artifact(str(path), artifact_path="validation")
        validation_probabilities[name] = scores
        results.append({
            "candidate": name, "sampling_strategy": ratio,
            "scale_pos_weight": weight, "threshold": THRESHOLD,
            **metrics, "fit_seconds": fit_seconds, "run_id": run.info.run_id,
        })
    print(f"Finished {name}: AP={metrics['average_precision']:.4f}, "
          f"F1={metrics['f1']:.4f}", flush=True)
    del pipeline


Training: weighted_baseline


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 14:36:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished weighted_baseline: AP=0.2067, F1=0.2677
Training: smote_0.25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 14:36:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished smote_0.25: AP=0.2028, F1=0.0020
Training: smote_0.50


...

Finished smote_0.50: AP=0.1967, F1=0.0080
Training: smote_0.75


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 14:38:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished smote_0.75: AP=0.1900, F1=0.0185
Training: smote_1.00


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/29 14:38:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished smote_1.00: AP=0.1843, F1=0.0321


In [6]:
results_df = pd.DataFrame(results).sort_values(
    ["average_precision", "f1"], ascending=False,
).reset_index(drop=True)
# Convenience export only; MLflow remains the experiment record.
results_df.to_csv(ROOT / "data/xgboost_smote_results.csv", index=False)
results_df


,candidate,sampling_strategy,scale_pos_weight,threshold,average_precision,f1,precision,recall,roc_auc,balanced_accuracy,accuracy,fit_seconds,run_id
0,weighted_baseline,NaN,7,0.5,0.206652,0.267731,0.184696,0.486410,0.684104,0.625042,0.736183,13.748953,3998d60ded25490abd7605866a669ba3
1,smote_0.25,0.25,1,0.5,0.202790,0.001956,0.571429,0.000979,0.681623,0.500449,0.900873,18.718726,23fda9028f564f8392f4ec97568c1a1e
2,smote_0.50,0.50,1,0.5,0.196662,0.008025,0.589286,0.004040,0.677094,0.501865,0.900970,20.589670,8b050e89afab40a79bea3ba4a6b328be
3,smote_0.75,0.75,1,0.5,0.189988,0.018476,0.461078,0.009427,0.672759,0.504107,0.900691,39.607889,811758a66284488fb3f1ee0387dfaa7e
4,smote_1.00,1.00,1,0.5,0.184317,0.032118,0.377410,0.016773,0.667731,0.506864,0.899768,27.530260,7784c5d05ffa4cf28c7d919596bea8ac


## What to inspect next

Expect five rows. Compare average precision (ranking quality) and F1, precision, and recall at the shared 0.5 threshold. The weighted baseline should be close to the previous F1 0.267731 and average precision 0.206652 if data and library behavior are unchanged.

Do not declare a winner from threshold 0.5 alone: different imbalance methods may need different thresholds. Next, reuse `validation_probabilities` (also saved in MLflow) for the same small validation threshold grid across all candidates; no refitting is needed. No improvement is guaranteed. Keep the test set out of this selection process.
